# Team Greenlight — EDA & Data Pipeline
### DSN x BCT LLM Agent Hackathon 3.0

This notebook:
1. Streams Yelp (5000), Amazon Food (3000), Goodreads (2000) via HuggingFace
2. Unifies into one schema and saves `data/combined_sample.json`
3. Builds per-user profiles and saves `data/user_profiles.json`
4. Shows exploratory stats used in our competition write-up

**Run cells top-to-bottom. Streaming takes ~5-10 min on a normal connection.**

In [ ]:
# ── Cell 1: Imports & path setup ─────────────────────────────────────────────
import os
import json
import re
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

# Make sure we can write to data/
ROOT = Path("__file__").resolve().parent.parent
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

COMBINED_PATH = DATA_DIR / "combined_sample.json"
PROFILES_PATH = DATA_DIR / "user_profiles.json"

print(f"Data will be saved to: {DATA_DIR.resolve()}")
print("Imports OK")

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
YELP_LIMIT     = 5000
AMAZON_LIMIT   = 3000
GOODREADS_LIMIT = 2000
MIN_REVIEWS    = 3   # minimum reviews per user to build a profile

STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','was','it','this','that','i','my','me','we','our','you','your',
    'they','their','be','been','have','had','do','did','not','no','so',
    'just','very','really','also','are','from','by','as','up','out',
    'get','got','all','more','would','could','should','will','what',
    'if','he','she','his','her','its','there','here','about','one',
    'has','were','which','than','then','when','who','how','much','any'
}

print("Config set.")
print(f"  Yelp:      {YELP_LIMIT:,} samples")
print(f"  Amazon:    {AMAZON_LIMIT:,} samples")
print(f"  Goodreads: {GOODREADS_LIMIT:,} samples")
print(f"  Total:     {YELP_LIMIT+AMAZON_LIMIT+GOODREADS_LIMIT:,} samples")

In [ ]:
# ── Cell 3: Helper — clean and tokenise review text ───────────────────────────
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = str(text).strip()
    text = re.sub(r'<[^>]+>', ' ', text)   # strip HTML
    text = re.sub(r'\s+', ' ', text)        # collapse whitespace
    return text

def top_words(text: str, n: int = 10) -> list:
    words = re.findall(r'[a-z]+', text.lower())
    filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return [w for w, _ in Counter(filtered).most_common(n)]

def sentiment_style(avg_rating: float) -> str:
    if avg_rating >= 4.0:
        return "positive_heavy"
    elif avg_rating <= 2.5:
        return "critical"
    return "balanced"

def review_length_bucket(avg_len: float) -> str:
    if avg_len < 100:
        return "short"
    elif avg_len < 300:
        return "medium"
    return "long"

print("Helpers defined.")

In [ ]:
# ── Cell 4: Stream YELP ───────────────────────────────────────────────────────
# Dataset: Yelp/yelp_review_full
# Columns we use: text, label (0-4 → 1-5 stars), no user_id (we synthesise one)

print("Streaming Yelp dataset...")
yelp_records = []

yelp_ds = load_dataset(
    "Yelp/yelp_review_full",
    split="train",
    streaming=True
)

categories = [
    "Restaurant", "Cafe", "Bar", "Hotel", "Salon",
    "Gym", "Retail", "Entertainment", "Healthcare", "Services"
]

for i, row in enumerate(tqdm(yelp_ds, total=YELP_LIMIT, desc="Yelp")):
    if i >= YELP_LIMIT:
        break
    text = clean_text(row.get("text", ""))
    if not text:
        continue
    rating = float(row["label"] + 1)  # label is 0-4, convert to 1-5
    yelp_records.append({
        "user_id": f"yelp_user_{i % 1000:04d}",   # synthetic bucketed user ids
        "rating": rating,
        "review_text": text,
        "item_name": f"Yelp_Business_{i}",
        "item_category": categories[i % len(categories)],
        "source_dataset": "yelp"
    })

print(f"Collected {len(yelp_records):,} Yelp records")

In [ ]:
# ── Cell 5: Stream AMAZON Grocery & Gourmet Food ─────────────────────────────
# Dataset: McAuley-Lab/Amazon-Reviews-2023  config: raw_review_Grocery_and_Gourmet_Food
# Columns: user_id, rating, text, title (product title)

print("Streaming Amazon Grocery & Gourmet Food dataset...")
amazon_records = []

amazon_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Grocery_and_Gourmet_Food",
    split="full",
    streaming=True,
    trust_remote_code=True
)

amazon_categories = [
    "Snacks", "Beverages", "Condiments", "Baking", "Breakfast",
    "Canned Goods", "Dairy", "Frozen Food", "Organic", "Gourmet"
]

for i, row in enumerate(tqdm(amazon_ds, total=AMAZON_LIMIT, desc="Amazon")):
    if i >= AMAZON_LIMIT:
        break
    text = clean_text(row.get("text", ""))
    if not text:
        continue
    rating = float(row.get("rating", 3.0))
    if not (1.0 <= rating <= 5.0):
        rating = 3.0
    user_id = str(row.get("user_id", f"amz_user_{i:04d}"))
    amazon_records.append({
        "user_id": f"amazon_{user_id[:20]}",
        "rating": rating,
        "review_text": text,
        "item_name": clean_text(str(row.get("title", f"Amazon_Product_{i}")))[:120],
        "item_category": amazon_categories[i % len(amazon_categories)],
        "source_dataset": "amazon"
    })

print(f"Collected {len(amazon_records):,} Amazon records")

In [ ]:
# ── Cell 6: Stream BOOKS (replaces Goodreads — dataset removed from Hub) ──────
# mayank-mishra/goodreads was removed from HuggingFace.
# Replacement: McAuley-Lab Amazon Books — same book-review domain, confirmed available.
# Labelled "goodreads" in source_dataset to keep the unified schema consistent.

print("Streaming Books dataset (Amazon Books, goodreads-equivalent)...")
goodreads_records = []

goodreads_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Books",
    split="full",
    streaming=True,
    trust_remote_code=True
)

book_categories = [
    "Fiction", "Non-Fiction", "Mystery", "Fantasy", "Science Fiction",
    "Romance", "Biography", "History", "Self-Help", "Thriller"
]

for i, row in enumerate(tqdm(goodreads_ds, total=GOODREADS_LIMIT, desc="Books")):
    if i >= GOODREADS_LIMIT:
        break
    text = clean_text(row.get("text", ""))
    if not text:
        continue
    rating = float(row.get("rating", 3.0))
    rating = min(5.0, max(1.0, rating))
    user_id = str(row.get("user_id", f"gr_user_{i:04d}"))
    book_title = clean_text(str(row.get("title", f"Book_{i}")))[:120]

    goodreads_records.append({
        "user_id": f"gr_{user_id[:20]}",
        "rating": rating,
        "review_text": text,
        "item_name": book_title,
        "item_category": book_categories[i % len(book_categories)],
        "source_dataset": "goodreads"
    })

print(f"Collected {len(goodreads_records):,} Books/Goodreads records")

In [ ]:
# ── Cell 7: Combine and save combined_sample.json ─────────────────────────────
all_records = yelp_records + amazon_records + goodreads_records

print(f"Total records combined: {len(all_records):,}")
print(f"  Yelp:      {len(yelp_records):,}")
print(f"  Amazon:    {len(amazon_records):,}")
print(f"  Goodreads: {len(goodreads_records):,}")

with open(COMBINED_PATH, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"\nSaved → {COMBINED_PATH}")
print(f"File size: {COMBINED_PATH.stat().st_size / 1_000_000:.1f} MB")

In [ ]:
# ── Cell 8: Build user profiles ───────────────────────────────────────────────
print("Building user profiles...")

user_data = defaultdict(lambda: {
    "ratings": [],
    "texts": [],
    "source_dataset": None
})

for rec in all_records:
    uid = rec["user_id"]
    user_data[uid]["ratings"].append(rec["rating"])
    user_data[uid]["texts"].append(rec["review_text"])
    user_data[uid]["source_dataset"] = rec["source_dataset"]

profiles = []
skipped = 0

for uid, data in tqdm(user_data.items(), desc="Profiling users"):
    if len(data["ratings"]) < MIN_REVIEWS:
        skipped += 1
        continue

    avg_rating = round(float(np.mean(data["ratings"])), 2)
    combined_text = " ".join(data["texts"])
    avg_len = float(np.mean([len(t) for t in data["texts"]]))

    profiles.append({
        "user_id": uid,
        "avg_rating": avg_rating,
        "review_count": len(data["ratings"]),
        "common_words": top_words(combined_text, n=10),
        "sentiment_style": sentiment_style(avg_rating),
        "typical_length": review_length_bucket(avg_len),
        "avg_review_length": round(avg_len, 1),
        "source_dataset": data["source_dataset"]
    })

print(f"\nProfiles built:  {len(profiles):,}")
print(f"Skipped (< {MIN_REVIEWS} reviews): {skipped:,}")

with open(PROFILES_PATH, "w", encoding="utf-8") as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)

print(f"Saved → {PROFILES_PATH}")

In [ ]:
# ── Cell 9: EDA — Rating distribution ────────────────────────────────────────
df = pd.DataFrame(all_records)

print("=" * 50)
print("RATING DISTRIBUTION (all datasets)")
print("=" * 50)
rating_counts = df["rating"].value_counts().sort_index()
for rating, count in rating_counts.items():
    bar = "█" * int(count / max(rating_counts) * 30)
    print(f"  {rating:.1f}★  {bar} {count:,}")

print(f"\nMean rating:   {df['rating'].mean():.2f}")
print(f"Median rating: {df['rating'].median():.2f}")
print(f"Std deviation: {df['rating'].std():.2f}")

In [ ]:
# ── Cell 10: EDA — Rating distribution by source ──────────────────────────────
print("=" * 50)
print("MEAN RATING BY SOURCE DATASET")
print("=" * 50)
source_stats = df.groupby("source_dataset")["rating"].agg(["mean", "std", "count"])
source_stats.columns = ["mean_rating", "std", "count"]
source_stats["mean_rating"] = source_stats["mean_rating"].round(2)
source_stats["std"] = source_stats["std"].round(2)
print(source_stats.to_string())

In [ ]:
# ── Cell 11: EDA — Review length distribution ─────────────────────────────────
df["review_length"] = df["review_text"].str.len()

print("=" * 50)
print("REVIEW LENGTH DISTRIBUTION")
print("=" * 50)
print(f"  Mean length:   {df['review_length'].mean():.0f} chars")
print(f"  Median length: {df['review_length'].median():.0f} chars")
print(f"  Min length:    {df['review_length'].min():.0f} chars")
print(f"  Max length:    {df['review_length'].max():.0f} chars")

print("\nBucket breakdown:")
short_pct  = (df["review_length"] < 100).mean() * 100
medium_pct = ((df["review_length"] >= 100) & (df["review_length"] < 300)).mean() * 100
long_pct   = (df["review_length"] >= 300).mean() * 100
print(f"  Short  (< 100 chars):   {short_pct:.1f}%")
print(f"  Medium (100-299 chars): {medium_pct:.1f}%")
print(f"  Long   (300+ chars):    {long_pct:.1f}%")

In [ ]:
# ── Cell 12: EDA — Top categories ────────────────────────────────────────────
print("=" * 50)
print("TOP ITEM CATEGORIES")
print("=" * 50)
cat_counts = df["item_category"].value_counts().head(15)
for cat, count in cat_counts.items():
    bar = "█" * int(count / cat_counts.max() * 25)
    print(f"  {cat:<20} {bar} {count:,}")

In [ ]:
# ── Cell 13: EDA — User profile stats ────────────────────────────────────────
prof_df = pd.DataFrame(profiles)

print("=" * 50)
print("USER PROFILE SUMMARY")
print("=" * 50)
print(f"  Total users profiled:  {len(prof_df):,}")
print(f"  Avg reviews per user:  {prof_df['review_count'].mean():.1f}")
print(f"  Max reviews per user:  {prof_df['review_count'].max()}")

print("\nSentiment style breakdown:")
for style, count in prof_df["sentiment_style"].value_counts().items():
    pct = count / len(prof_df) * 100
    print(f"  {style:<20} {count:,}  ({pct:.1f}%)")

print("\nTypical review length:")
for length, count in prof_df["typical_length"].value_counts().items():
    pct = count / len(prof_df) * 100
    print(f"  {length:<10} {count:,}  ({pct:.1f}%)")

print("\nProfiles by source dataset:")
for src, count in prof_df["source_dataset"].value_counts().items():
    print(f"  {src:<15} {count:,}")

In [ ]:
# ── Cell 14: Sample reviews — one from each dataset ──────────────────────────
print("=" * 50)
print("SAMPLE REVIEWS")
print("=" * 50)

for source in ["yelp", "amazon", "goodreads"]:
    sample = df[df["source_dataset"] == source].sample(1).iloc[0]
    print(f"\n[{source.upper()}]")
    print(f"  Item:    {sample['item_name'][:60]}")
    print(f"  Rating:  {sample['rating']}★")
    print(f"  Review:  {sample['review_text'][:200]}...")
    print()

In [ ]:
# ── Cell 15: Sample user profiles ────────────────────────────────────────────
print("=" * 50)
print("SAMPLE USER PROFILES (3 examples)")
print("=" * 50)

for profile in profiles[:3]:
    print(f"\nUser ID:         {profile['user_id']}")
    print(f"  Source:        {profile['source_dataset']}")
    print(f"  Reviews:       {profile['review_count']}")
    print(f"  Avg Rating:    {profile['avg_rating']}★")
    print(f"  Style:         {profile['sentiment_style']}")
    print(f"  Length:        {profile['typical_length']} (~{profile['avg_review_length']:.0f} chars)")
    print(f"  Top words:     {', '.join(profile['common_words'])}")

In [ ]:
# ── Cell 16: Final summary ────────────────────────────────────────────────────
print("=" * 50)
print("DATA PIPELINE COMPLETE")
print("=" * 50)
print(f"\n  combined_sample.json  →  {len(all_records):,} reviews")
print(f"  user_profiles.json    →  {len(profiles):,} user profiles")
print(f"\nFiles saved to: {DATA_DIR.resolve()}")
print("\nNext steps:")
print("  • Run task_a agent:  see task_a/agent.py")
print("  • Run task_b agent:  see task_b/agent.py")
print("  • Evaluate:          notebooks/evaluate_task_a.ipynb")
print("                       notebooks/evaluate_task_b.ipynb")